In [1]:
import bibtexparser
import feedparser
import os
import re
import requests
import pandas as pd
import urllib.request as libreq
from urllib.parse import quote

In [2]:
from dotenv import load_dotenv
load_dotenv()

SCOPUS_API_KEY = os.getenv("SCOPUS_API_KEY")  # Required for Scopus API access

In [3]:
# ACM -  Within The ACM Guide to Computing Literature (3,937,257 records)
'("language model*" OR llm*) AND ("knowledge graph*" OR kg*) AND (hallucinat*)'

# IEEE -  Does not allow wildcards for 2 or less characters
'("Abstract":"language model*" OR llm*) AND ("Abstract":"knowledge graph*" OR kg) AND ("Abstract":hallucinat*)'

# Springer
'("language model*" OR llm* OR encoder* OR decoder* OR gpt OR rag) AND (knowledge* OR graph* OR kg*) AND (hallucinat* OR fact* OR qa OR trust* OR coheren* OR reliab* OR explain*)'

'("language model*" OR llm* OR encoder* OR decoder* OR gpt OR rag) AND (knowledge* OR graph* OR kg*) AND (hallucinat* OR fact* OR qa OR trust* OR coheren* OR reliab* OR explain*)'

In [4]:
dblp_query = "llm|language model|encoder|decoder|gpt|rag kg|knowledge|graph hallucinat|fact|qa|trust|coheren|reliab|explain"

acl_llm_keywords = "llm|language model"
acl_kg_keywords = "knowledge graph|kg"  # KG due to string.contains may retrieve more false positives (e.g., bacKGround)
acl_hallucination_keywords = "hallucinat"

arxiv_query = 'abs:"language model" AND "knowledge graph" AND hallucination'

scopus_query = 'TITLE-ABS-KEY("language model*" OR llm* AND "knowledge graph*" OR kg* AND hallucinat*)'

### DBLP

In [7]:
def search_dblp(query, max_results=5):
    """
    Search for publications on DBLP and return metadata for each result.
    """
    url = "https://dblp.org/search/publ/api"
    params = {"q": query, "format": "json", "h": max_results}
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    hits = data.get("result", {}).get("hits", {}).get("hit", [])
    if not hits:
        print("No results found on DBLP.")
        return []

    results = []
    for hit in hits:
        info = hit.get("info", {})
        authors = info.get("authors", {}).get("author")
        if isinstance(authors, dict):
            authors = [authors.get("text")]
        elif isinstance(authors, list):
            authors = [a.get("text") if isinstance(a, dict) else a for a in authors]

        entry = {
            "title": info.get("title"),
            "authors": authors or [],
            "year": info.get("year"),
            "venue": info.get("venue"),
            "type": info.get("type"),
            "url": info.get("url"),
            "doi": info.get("doi"),
            "dblp_key": info.get("key"),
        }
        results.append(entry)

    print(f"Found {len(results)} papers.")

    return results

def get_crossref_abstract(doi):
    """
    Fetch the abstract for a given DOI from the Crossref API.
    """
    if not doi:
        return None
    url = f"https://api.crossref.org/works/{doi}"
    response = requests.get(url)
    if response.status_code != 200:
        return None

    data = response.json().get("message", {})
    abstract = data.get("abstract")
    if abstract:
        clean = re.sub(r"<.*?>", "", abstract)  # strip XML/HTML tags
        return clean.strip()
    return None

def get_full_metadata_with_abstract(query, max_results=3):
    """
    DBLP → metadata + Crossref → abstract.
    """
    print(f"Searching DBLP for: '{query}' ...")
    papers = search_dblp(query, max_results=max_results)
    if not papers:
        return []
    
    print(f"Enriching with abstracts ...")

    enriched = []
    for p in papers:
        abstract = get_crossref_abstract(p.get("doi"))
        p["abstract"] = abstract or "No abstract found."
        enriched.append(p)

    return enriched

In [8]:
# DBLP API provides autocompletion to the search terms.
# Source: https://dblp.org/faq/How+to+use+the+dblp+search+API.html
# Info on search string: https://dblp.org/faq/1474589.html

results = search_dblp(dblp_query, max_results=1000)

Found 246 papers.


In [9]:
for result in results[:5]:  # Print first 5 results
    print(f"{result['year']} - {result['title']}")
    print("-" * 30)

2024 - Give us the Facts: Enhancing Large Language Models With Knowledge Graphs for Fact-Aware Language Modeling.
------------------------------
2025 - RAG-KG-IL: A Multi-Agent Hybrid Framework for Reducing Hallucinations and Enhancing LLM Reasoning through RAG and Incremental Knowledge Graph Learning Integration.
------------------------------
2024 - Graph Hierarchy and Language Model-Based Explainable Entity Alignment of Knowledge Graphs.
------------------------------
2023 - ChatGPT is not Enough: Enhancing Large Language Models with Knowledge Graphs for Fact-aware Language Modeling.
------------------------------
2026 - Integrating knowledge from knowledge graphs and large language models for explainable entity alignment.
------------------------------


In [10]:
dblp_df = pd.DataFrame(results)

In [11]:
# Remove square parentheses from column authors
dblp_df['authors'] = dblp_df['authors'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

In [12]:
dblp_df.head(2)

,title,authors,year,venue,type,url,doi,dblp_key
0,Give us the Facts: Enhancing Large Language Mo...,"Linyao Yang, Hongyang Chen 0001, Zhao Li 0007,...",2024,IEEE Trans. Knowl. Data Eng.,Journal Articles,https://dblp.org/rec/journals/tkde/YangCLDW24,10.1109/TKDE.2024.3360454,journals/tkde/YangCLDW24
1,RAG-KG-IL: A Multi-Agent Hybrid Framework for ...,"Hong Qing Yu, Frank McQuade",2025,CoRR,Informal and Other Publications,https://dblp.org/rec/journals/corr/abs-2503-13514,10.48550/ARXIV.2503.13514,journals/corr/abs-2503-13514


In [13]:
# Filter only papers from 2020 onwards
dblp_df_2020_onwards = dblp_df[dblp_df['year'].astype(int) >= 2020].reset_index(drop=True)
dblp_df_2020_onwards.head(2)

,title,authors,year,venue,type,url,doi,dblp_key
0,Give us the Facts: Enhancing Large Language Mo...,"Linyao Yang, Hongyang Chen 0001, Zhao Li 0007,...",2024,IEEE Trans. Knowl. Data Eng.,Journal Articles,https://dblp.org/rec/journals/tkde/YangCLDW24,10.1109/TKDE.2024.3360454,journals/tkde/YangCLDW24
1,RAG-KG-IL: A Multi-Agent Hybrid Framework for ...,"Hong Qing Yu, Frank McQuade",2025,CoRR,Informal and Other Publications,https://dblp.org/rec/journals/corr/abs-2503-13514,10.48550/ARXIV.2503.13514,journals/corr/abs-2503-13514


In [14]:
# Rename columns for clarity
dblp_df_2020_onwards.rename(columns={
    'title': 'Title',
    'authors': 'Authors',
    'year': 'Year',
    'venue': 'Venue',
    'type': 'Type',
    'url': 'URL',
    'doi': 'DOI',
    'dblp_key': 'DBLP Key'
}, inplace=True)

dblp_df_2020_onwards.head(2)

,Title,Authors,Year,Venue,Type,URL,DOI,DBLP Key
0,Give us the Facts: Enhancing Large Language Mo...,"Linyao Yang, Hongyang Chen 0001, Zhao Li 0007,...",2024,IEEE Trans. Knowl. Data Eng.,Journal Articles,https://dblp.org/rec/journals/tkde/YangCLDW24,10.1109/TKDE.2024.3360454,journals/tkde/YangCLDW24
1,RAG-KG-IL: A Multi-Agent Hybrid Framework for ...,"Hong Qing Yu, Frank McQuade",2025,CoRR,Informal and Other Publications,https://dblp.org/rec/journals/corr/abs-2503-13514,10.48550/ARXIV.2503.13514,journals/corr/abs-2503-13514


In [15]:
# Print number of papers after filtering
print(f"Number of papers from 2020 onwards: {len(dblp_df_2020_onwards)}")

Number of papers from 2020 onwards: 240


In [16]:
dblp_df_2020_onwards.Type.unique()

array(['Journal Articles', 'Informal and Other Publications',
       'Conference and Workshop Papers'], dtype=object)

In [17]:
print(f"Number of informal papers: {len(dblp_df_2020_onwards[dblp_df_2020_onwards['Type'] == 'Informal and Other Publications'])}")

Number of informal papers: 120


In [ ]:
# Remove informal papers
dblp_filtered = dblp_df_2020_onwards[dblp_df_2020_onwards['Type'] != 'Informal and Other Publications'].reset_index(drop=True)
print(f"Number of papers after removing informal ones: {len(dblp_filtered)}")

Number of papers after removing informal ones: 120


,Title,Authors,Year,Venue,Type,URL,DOI,DBLP Key
0,Give us the Facts: Enhancing Large Language Mo...,"Linyao Yang, Hongyang Chen 0001, Zhao Li 0007,...",2024,IEEE Trans. Knowl. Data Eng.,Journal Articles,https://dblp.org/rec/journals/tkde/YangCLDW24,10.1109/TKDE.2024.3360454,journals/tkde/YangCLDW24
1,Graph Hierarchy and Language Model-Based Expla...,"Kunyoung Kim, Donggyu Kim, Mye M. Sohn",2024,EXPLAINS,Conference and Workshop Papers,https://dblp.org/rec/conf/explains/KimKS24,10.5220/0013012300003886,conf/explains/KimKS24


In [ ]:
dblp_filtered.to_pickle("../results/dblp_results.pkl")

### ACL

Load external pickle file to search within a dataframe.

In [ ]:
acl_df = pd.read_pickle("../data/acl_bibliography.pkl")

In [6]:
acl_df.head(2)

,bib_key,url,publisher,address,year,month,editor,title,ENTRYTYPE,ID,abstract,pages,booktitle,author,isbn,doi,volume,journal,language,number
0,yrrsds-2025-1,https://aclanthology.org/2025.yrrsds-1.0/,Association for Computational Linguistics,"Avignon, France",2025,August,"Whetten, Ryan and\nSucal, Virgile and\nNgo, ...",Proceedings of the 21st Workshop of Young Rese...,proceedings,yrrsds-2025-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chen-2025-research,https://aclanthology.org/2025.yrrsds-1.1/,Association for Computational Linguistics,"Avignon, France",2025,August,"Whetten, Ryan and\nSucal, Virgile and\nNgo, ...",Research on {LLM}s-Empowered Conversational {A...,inproceedings,chen-2025-research,"This is a position paper for my research, incl...",1--3,Proceedings of the 21st Workshop of Young Rese...,"Chen, Ben",NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
acl_df.year.unique()

array(['2025', '2024', '2023', '2022', '2021', '2020'], dtype=object)

In [8]:
acl_df.language.unique()

array([nan, 'fra', 'ita', 'zho', 'eng', 'por', 'pt-BR'], dtype=object)

In [9]:
# Remove where language is fra or zho
acl_df = acl_df[~acl_df['language'].isin(['fra', 'zho', 'ita', 'por', 'pt-BR'])]

In [10]:
# Remove columsn that are not needed
acl_synthesis = acl_df.drop(columns=['editor', 'ENTRYTYPE', 'ID', 'pages', 'isbn', 'volume', 'language', 'number', 'address', 'month'])

In [11]:
# Rename columns for clarity
acl_synthesis.rename(columns={
    'title': 'Title',
    'author': 'Authors',
    'year': 'Year',
    'booktitle': 'Venue',
    'journal': 'Journal',
    'bib_key': 'ACL Key',
    'url': 'URL',
    'doi': 'DOI',
    'abstract': 'Abstract',
    'publisher': 'Publisher'
}, inplace=True)

In [12]:
acl_synthesis.head(2)

,ACL Key,URL,Publisher,Year,Title,Abstract,Venue,Authors,DOI,Journal
0,yrrsds-2025-1,https://aclanthology.org/2025.yrrsds-1.0/,Association for Computational Linguistics,2025,Proceedings of the 21st Workshop of Young Rese...,NaN,NaN,NaN,NaN,NaN
1,chen-2025-research,https://aclanthology.org/2025.yrrsds-1.1/,Association for Computational Linguistics,2025,Research on {LLM}s-Empowered Conversational {A...,"This is a position paper for my research, incl...",Proceedings of the 21st Workshop of Young Rese...,"Chen, Ben",NaN,NaN


In [13]:
# Count rows where abstract is NaN
len(acl_synthesis[acl_synthesis['Abstract'].isna()])

4453

In [14]:
def search_papers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Search ACL papers by filtering abstracts for language models, 
    factuality/hallucination, and knowledge graphs.
    """
    mask = (
        df['Abstract'].str.contains(f"({acl_llm_keywords})", case=False) &
        df['Abstract'].str.contains(f"({acl_kg_keywords})", case=False) &
        df['Abstract'].str.contains(f"({acl_hallucination_keywords})", case=False)
    )
    return df[mask]

In [15]:
acl_results = search_papers(acl_synthesis)

/tmp/ipykernel_8419/1172233082.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['Abstract'].str.contains(f"({acl_llm_keywords})", case=False) &
/tmp/ipykernel_8419/1172233082.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['Abstract'].str.contains(f"({acl_kg_keywords})", case=False) &
/tmp/ipykernel_8419/1172233082.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['Abstract'].str.contains(f"({acl_hallucination_keywords})", case=False)


In [16]:
# Print number of results
print(f"Number of ACL papers matching criteria: {len(acl_results)}")

Number of ACL papers matching criteria: 69


In [17]:
acl_results.head(2)

,ACL Key,URL,Publisher,Year,Title,Abstract,Venue,Authors,DOI,Journal
1165,chen-etal-2025-tcqa2,https://aclanthology.org/2025.realm-1.20/,Association for Computational Linguistics,2025,{TCQA}$^2$: A Tiered Conversational {Q}{\&}{A}...,This paper focuses on intelligent Q{\&}A assis...,Proceedings of the 1st Workshop for Research o...,"Chen, Ze and\nWei, Chengcheng and\nZheng, Ji...",10.18653/v1/2025.realm-1.20,NaN
1965,zhu-etal-2025-knowledge,https://aclanthology.org/2025.naacl-long.449/,Association for Computational Linguistics,2025,Knowledge Graph-Guided Retrieval Augmented Gen...,Retrieval-augmented generation (RAG) has emerg...,Proceedings of the 2025 Conference of the Nati...,"Zhu, Xiangrong and\nXie, Yuexiang and\nLiu, ...",10.18653/v1/2025.naacl-long.449,NaN


In [18]:
# Parse the Title column to remove bibtex syntax like curly braces {}
acl_results.loc[:, 'Title'] = acl_results['Title'].str.replace(r"[{}]", "", regex=True)
# Remove newlines from Authors
acl_results.loc[:, 'Authors'] = acl_results['Authors'].str.replace(r"\n", " ", regex=True)

In [ ]:
# Save to pickle
acl_results.to_pickle("../results/acl_results.pkl")

### ACM

Literature search performed through the website interface. Parsing takes as input the web interface export saved under `data`.

In [5]:
useful_fields = [
    "title",
    "author",
    "year",
    "booktitle",
    "journal",
    "publisher",
    "abstract",
    "doi",
    "url"
]

In [6]:
def load_bibtex(file_path: str) -> pd.DataFrame:
    with open(file_path, 'r') as bibtex_file:
        bib_database = bibtexparser.load(bibtex_file)

    print("Number of Papers:", len(bib_database.entries_dict.keys()))

    # Load the dictionary into a pandas dataframe
    df = pd.DataFrame.from_dict(bib_database.entries_dict, orient='index')

    return df

In [7]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # Print information about years and languages
    print(df.year.unique())

    # Remove when year is < 2020
    df = df[df['year'].astype(int) >= 2020]

    # Keep only useful fields
    df = df[[col for col in useful_fields if col in df.columns]]

    # Rename columns for consistency
    renamed_df = df.rename(columns={
        'title': 'Title',
        'author': 'Authors',
        'year': 'Year',
        'booktitle': 'Venue',
        'journal': 'Journal',
        'url': 'URL',
        'doi': 'DOI',
        'abstract': 'Abstract',
        'publisher': 'Publisher'
    }).reset_index(drop=True)

    return renamed_df

In [8]:
acm_df = load_bibtex('../data/acm.bib')
acm_df.head(2)

Number of Papers: 81


,series,location,keywords,numpages,pages,booktitle,abstract,doi,url,address,...,ENTRYTYPE,ID,month,journal,issn,number,volume,issue_date,articleno,note
10.1145/3756580.3756654,ICEKIM '25,,"intelligent question answering, knowledge grap...",10,446–455,Proceedings of the 2025 6th International Conf...,"At present, the data in the aerospace science ...",10.1145/3756580.3756654,https://doi.org/10.1145/3756580.3756654,"New York, NY, USA",...,inproceedings,10.1145/3756580.3756654,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10.1145/3746252.3760893,CIKM '25,"Seoul, Republic of Korea","knowledge graph, large language models, neighb...",5,5495–5499,Proceedings of the 34th ACM International Conf...,Existing approaches to leveraging knowledge gr...,10.1145/3746252.3760893,https://doi.org/10.1145/3746252.3760893,"New York, NY, USA",...,inproceedings,10.1145/3746252.3760893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
clean_acm_df = clean_dataframe(acm_df)
clean_acm_df.head(2)

<StringArray>
['2025', '2024', '2023']
Length: 3, dtype: str


,Title,Authors,Year,Venue,Journal,Publisher,Abstract,DOI,URL
0,A Question Answering System for Aerospace Larg...,"Chen, Fei and Wen, Zhonghua and Liu, Bo",2025,Proceedings of the 2025 6th International Conf...,NaN,Association for Computing Machinery,"At present, the data in the aerospace science ...",10.1145/3756580.3756654,https://doi.org/10.1145/3756580.3756654
1,SAKG: Structure-Aware Large Language Model Fra...,"Zhang, Qingyu and Hu, Min and Fei, Wenlong and...",2025,Proceedings of the 34th ACM International Conf...,NaN,Association for Computing Machinery,Existing approaches to leveraging knowledge gr...,10.1145/3746252.3760893,https://doi.org/10.1145/3746252.3760893


In [10]:
# Save to pickle
clean_acm_df.to_pickle("../results/acm_results.pkl")

### IEEE

Literature search performed through the website interface. Parsing takes as input the web interface export saved under `data`.

In [11]:
ieee_df = load_bibtex('../data/ieee.bib')
ieee_df.head(2)

Number of Papers: 74


,url,isbn,publisher,issn,doi,keywords,abstract,pages,number,volume,year,booktitle,author,ENTRYTYPE,ID,month,title,journal
11099031,https://ieeexplore.ieee.org/document/11099031,9781836206224,Packt Publishing,,,,A comprehensive guide to building cutting-edge...,,,,2025,Building Neo4j-Powered Applications with LLMs:...,"Anthapu, Ravindranatha and Agarwal, Siddhant a...",book,11099031,NaN,NaN,NaN
10965196,NaN,NaN,NaN,,10.1109/ICIPNP62754.2023.00095,Training;Visualization;Analytical models;Large...,The visualization model using LLM primarily op...,429-434,,,2023,2023 International Conference on Information P...,"Yang, Wenzhou and Lv, Wenzhe",inproceedings,10965196,Oct,Visualization Over Large Language Model Using ...,NaN


In [12]:
clean_ieee_df = clean_dataframe(ieee_df)
clean_ieee_df.head(2)

<StringArray>
['2025', '2023', '2024']
Length: 3, dtype: str


,Title,Authors,Year,Venue,Journal,Publisher,Abstract,DOI,URL
0,NaN,"Anthapu, Ravindranatha and Agarwal, Siddhant a...",2025,Building Neo4j-Powered Applications with LLMs:...,NaN,Packt Publishing,A comprehensive guide to building cutting-edge...,,https://ieeexplore.ieee.org/document/11099031
1,Visualization Over Large Language Model Using ...,"Yang, Wenzhou and Lv, Wenzhe",2023,2023 International Conference on Information P...,NaN,NaN,The visualization model using LLM primarily op...,10.1109/ICIPNP62754.2023.00095,NaN


In [13]:
# Save to pickle
clean_ieee_df.to_pickle("../results/ieee_results.pkl")

### Springer

Literature search performed through the website interface. Parsing takes as input the web interface export saved under `data`.

In [14]:
springer_df = pd.read_csv('../data/springer.csv')
springer_df.head(2)

,Item Title,Publication Title,Book Series Title,Journal Volume,Journal Issue,Item DOI,Authors,Publication Year,URL,Content Type
0,Interactive and&#xa0;Provenance-Aware Search a...,New Trends in Theory and Practice of Digital L...,NaN,NaN,NaN,10.1007/978-3-032-06136-2_24,Iordanis SapidisValantis ZervosMichalis Mounta...,2026,https://link.springer.com/chapter/10.1007/978-...,Conference paper
1,XLR-KGDD: leveraging LLM and RAG for knowledge...,Knowledge and Information Systems,NaN,NaN,NaN,10.1007/s10115-025-02465-8,Punam BediAnjali ThukralShivani Dhiman,2025,https://link.springer.com/article/10.1007/s101...,Article


In [15]:
# Keep useful fields and rename columns
springer_df = springer_df.rename(columns={
    'Item Title': 'Title',
    'Publication Year': 'Year',
    'Publication Title': 'Venue',
    'Item DOI': 'DOI',
    'URL': 'URL',
    'Content Type': 'Type'
})
springer_df = springer_df[['Title', 'Authors', 'Year', 'Venue', 'DOI', 'URL', 'Type']]
springer_df.head(2)

,Title,Authors,Year,Venue,DOI,URL,Type
0,Interactive and&#xa0;Provenance-Aware Search a...,Iordanis SapidisValantis ZervosMichalis Mounta...,2026,New Trends in Theory and Practice of Digital L...,10.1007/978-3-032-06136-2_24,https://link.springer.com/chapter/10.1007/978-...,Conference paper
1,XLR-KGDD: leveraging LLM and RAG for knowledge...,Punam BediAnjali ThukralShivani Dhiman,2025,Knowledge and Information Systems,10.1007/s10115-025-02465-8,https://link.springer.com/article/10.1007/s101...,Article


In [16]:
print(springer_df['Year'].unique())
springer_df = springer_df[springer_df['Year'].astype(int) >= 2020].reset_index(drop=True)

[2026 2025 2023 2024]


In [17]:
# Save to pickle
springer_df.to_pickle("../results/springer_results.pkl")

### Scopus

In [38]:
def scopus_search(query, max_results=200):
    """
    Search Scopus with boolean logic inside titles and optionally abstracts.
    Returns parsed JSON results.
    """

    url = "https://api.elsevier.com/content/search/scopus"
    params = {
        "query": query,
        "count": max_results,
        "apiKey": SCOPUS_API_KEY,
        "view": "STANDARD"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()
    
    # Extract meaningful fields
    results = []
    entries = data.get("search-results", {}).get("entry", [])
    for e in entries:
        results.append({
            "title": e.get("dc:title"),
            "authors": e.get("dc:creator"),
            "doi": e.get("prism:doi"),
            "year": e.get("prism:coverDate", "")[:4],
            "scopus_id": e.get("dc:identifier"),
            "url": e.get("prism:url")
        })

    return results

Equivalent search on the website:

TITLE-ABS-KEY ( "language model*" OR llm* AND "knowledge graph*" OR kg* AND hallucinat* ) AND PUBYEAR > 2019 AND PUBYEAR < 2027 AND SUBJAREA ( COMP ) AND LANGUAGE ( english )

In [37]:
# Search within titles and abstracts and keywords
scopus_results_24 = scopus_search(f'{scopus_query}' \
'AND PUBYEAR > 2019 AND PUBYEAR < 2025 ' \
'AND SUBJAREA ( COMP ) AND LANGUAGE ( english )', max_results=200)

print('Numer of Scopus results 2020-2024:', len(scopus_results_24))

{'@_fa': 'true', 'link': [{'@_fa': 'true', '@ref': 'self', '@href': 'https://api.elsevier.com/content/abstract/scopus_id/85211914102'}, {'@_fa': 'true', '@ref': 'author-affiliation', '@href': 'https://api.elsevier.com/content/abstract/scopus_id/85211914102?field=author,affiliation'}, {'@_fa': 'true', '@ref': 'scopus', '@href': 'https://www.scopus.com/inward/record.uri?partnerID=HzOxMe3b&scp=85211914102&origin=inward'}, {'@_fa': 'true', '@ref': 'scopus-citedby', '@href': 'https://www.scopus.com/inward/citedby.uri?partnerID=HzOxMe3b&scp=85211914102&origin=inward'}], 'prism:url': 'https://api.elsevier.com/content/abstract/scopus_id/85211914102', 'dc:identifier': 'SCOPUS_ID:85211914102', 'eid': '2-s2.0-85211914102', 'dc:title': 'KG-EGV: A Framework for Question Answering with Integrated Knowledge Graphs and Large Language Models', 'dc:creator': 'Hou K.', 'prism:publicationName': 'Electronics Switzerland', 'prism:eIssn': '20799292', 'prism:volume': '13', 'prism:issueIdentifier': '23', 'pris

In [34]:
scopus_results_26 = scopus_search(f'{scopus_query}' \
'AND PUBYEAR > 2024 AND PUBYEAR < 2027 ' \
'AND SUBJAREA ( COMP ) AND LANGUAGE ( english )', max_results=200)

print('Numer of Scopus results 2025-2026:', len(scopus_results_26))

Numer of Scopus results 2025-2026: 165


In [35]:
# Merge the two lists
scopus_results = scopus_results_24 + scopus_results_26

# Load into a DataFrame
scopus_df = pd.DataFrame(scopus_results)
scopus_df.head(2)

,title,authors,doi,year,abstract,scopus_id,url
0,KG-EGV: A Framework for Question Answering wit...,Hou K.,10.3390/electronics13234835,2024,None,SCOPUS_ID:85211914102,https://api.elsevier.com/content/abstract/scop...
1,Augmented non-hallucinating large language mod...,Gilbert S.,10.1038/s41746-024-01081-0,2024,None,SCOPUS_ID:85191180518,https://api.elsevier.com/content/abstract/scop...


In [39]:
# Rename columns for clarity
scopus_df.rename(columns={
    'title': 'Title',
    'authors': 'Authors',
    'doi': 'DOI',
    'year': 'Year',
    'abstract': 'Abstract',
    'scopus_id': 'Scopus ID',
    'url': 'URL'
}, inplace=True)

In [ ]:
# Save to pickle
scopus_df.to_pickle("../results/scopus_results.pkl")

### arXiv

In [42]:
def arxiv_search(query, max_results=25):
    """
    Search arXiv by applying the query to both title and abstract.
    Boolean operators (AND, OR, ANDNOT) are supported.
    """

    with libreq.urlopen(f'http://export.arxiv.org/api/query?search_query={query}&start=0&max_results={max_results}') as url:
      data = url.read()

    feed = feedparser.parse(data)

    results = []
    for entry in feed.entries:
        results.append({
            "title": entry.title,
            "authors": [a.name for a in entry.authors],
            "summary": entry.summary,
            "published": entry.published,
            "updated": entry.updated,
            "pdf_url": next((l.href for l in entry.links if l.type == "application/pdf"), None),
            "arxiv_url": entry.link,
            "category": entry.arxiv_primary_category.get("term", "no")[:2]
        })

    return results

'ti:("language model" OR llm OR encoder OR decoder OR gpt OR rag) AND ' \
'ti:(kg OR "knowledge graph" OR "knowledge base" OR "graph database") AND ' \
'ti:(hallucinat OR fact OR trust OR reliab OR explain)'

In [43]:
arxiv_results = arxiv_search(quote(arxiv_query), max_results=1000)

In [44]:
len(arxiv_results)

301

In [46]:
arxiv_titles = [entry['title'] for entry in arxiv_results]
arxiv_titles[:5]

['Enhancing Knowledge Graph Construction: Evaluating with Emphasis on Hallucination, Omission, and Graph Similarity Metrics',
 'GraphEval: A Knowledge-Graph Based LLM Hallucination Evaluation Framework',
 'Assessing LLMs Suitability for Knowledge Graph Completion',
 'Mitigating Large Language Model Hallucinations via Autonomous Knowledge Graph-based Retrofitting',
 'From Hallucinations to Facts: Enhancing Language Models with Curated Knowledge Graphs']

In [47]:
# Add resutls to a dataframe
arxiv_df = pd.DataFrame(arxiv_results)
arxiv_df.head(2)

,title,authors,summary,published,updated,pdf_url,arxiv_url,category
0,Enhancing Knowledge Graph Construction: Evalua...,"[Hussam Ghanem, Christophe Cruz]",Recent advancements in large language models h...,2025-02-07T11:19:01Z,2025-02-07T11:19:01Z,https://arxiv.org/pdf/2502.05239v1,https://arxiv.org/abs/2502.05239v1,cs
1,GraphEval: A Knowledge-Graph Based LLM Halluci...,"[Hannah Sansford, Nicholas Richardson, Hermina...",Methods to evaluate Large Language Model (LLM)...,2024-07-15T15:11:16Z,2024-07-15T15:11:16Z,https://arxiv.org/pdf/2407.10793v1,https://arxiv.org/abs/2407.10793v1,cs


In [49]:
# Filter for category cs (computer science)
arxiv_cs_df = arxiv_df[arxiv_df['category'].str.startswith('cs')].reset_index(drop=True)
print(f"Number of arXiv CS papers matching criteria: {len(arxiv_cs_df)}")

Number of arXiv CS papers matching criteria: 300


In [51]:
# Extract year from updated date
arxiv_cs_df['updated_year'] = pd.to_datetime(arxiv_cs_df['updated']).dt.year

# Remove pdf_url column
arxiv_df_short = arxiv_cs_df[['title', 'authors', 'summary', 'updated_year', 'arxiv_url']]
arxiv_df_short.head(2)

,title,authors,summary,updated_year,arxiv_url
0,Enhancing Knowledge Graph Construction: Evalua...,"[Hussam Ghanem, Christophe Cruz]",Recent advancements in large language models h...,2025,https://arxiv.org/abs/2502.05239v1
1,GraphEval: A Knowledge-Graph Based LLM Halluci...,"[Hannah Sansford, Nicholas Richardson, Hermina...",Methods to evaluate Large Language Model (LLM)...,2024,https://arxiv.org/abs/2407.10793v1


In [52]:
# Rename columns for clarity
arxiv_df_short = arxiv_df_short.rename(columns={
    'title': 'Title',
    'authors': 'Authors',
    'summary': 'Abstract',
    'updated_year': 'Year',
    'arxiv_url': 'URL'
})

In [53]:
# Select where year is 2020 or later
arxiv_df_2020_onwards = arxiv_df_short[arxiv_df_short['Year'] >= 2020].reset_index(drop=True)
arxiv_df_2020_onwards.head(2)

,Title,Authors,Abstract,Year,URL
0,Enhancing Knowledge Graph Construction: Evalua...,"[Hussam Ghanem, Christophe Cruz]",Recent advancements in large language models h...,2025,https://arxiv.org/abs/2502.05239v1
1,GraphEval: A Knowledge-Graph Based LLM Halluci...,"[Hannah Sansford, Nicholas Richardson, Hermina...",Methods to evaluate Large Language Model (LLM)...,2024,https://arxiv.org/abs/2407.10793v1


In [54]:
len(arxiv_df_2020_onwards)

300

In [ ]:
# Save to pickle
arxiv_df_2020_onwards.to_pickle("../results/arxiv_results.pkl")